# SE-HOPE exclusive-instance CSV export

This is the final, mandatory CPU postprocessing step for the [full model notebook](https://www.kaggle.com/code/topabaem/filament-v02-submission-final-001).
The original CSV was rejected because predicted instances overlapped. The model checkpoint and thresholds are unchanged.
Masks are already in descending fused-confidence order. Each pixel belongs to the first mask containing it; later masks lose only already-owned pixels and empty residuals are dropped. This preserves the foreground union, creates no new pixels, and uses no annotations. No overlap thresholds are tuned.

Attach the completed final-001 notebook output and use **no accelerator**. A 20-minute alarm bounds this export. The immutable full training/inference notebook, code and provenance are at https://github.com/Topabaem05/solar-filament-sehope-2026 .


In [ ]:
USE_CURRENT_RUN = False


In [ ]:
import base64, hashlib, importlib.metadata, io, json, signal, sys, time, zipfile
from pathlib import Path
import numpy as np
from pycocotools import mask as coco_mask
started = time.monotonic()
def stop_on_deadline(*args):
    raise TimeoutError("CPU export time limit reached")
signal.signal(signal.SIGALRM, stop_on_deadline)
signal.setitimer(signal.ITIMER_REAL, 1200)
raw = base64.b64decode('UEsDBBQAAAAIADNiSV2pJPypfAYAAHYQAAAaAAAAc2NyaXB0cy9kaXNqb2ludF9leHBvcnQucHm1V1tv2zYUfvev4PIwSqijtcU2DAE8YBhaYC9FgRZ98QyCkWibi0RqJJXGy7Lfvu+QlCzbafc0PyQSee7nOxddXV29MVvrasXUQ90OXt8rpo0P0uCo1w+q9eyzDns7BFbvpdlps2Nhr1hnG9Uy6xi41c7ZwTRsMNqaarF4JwPJ+fXDJxA0yjHtWaN8rUxD7NvBq4bV1mx1o0hPAX23LWSq7la56zt1YEErduuUvCurxUeo22rnw9Ey+9l45vfSQVCysmLvLGvlLR6XTHdyp1hraxhiDQ5gp69hqGdbZ7uFNEzdy3aI18yrwJQJsDPso6V47LTRPugaZna9CpoIr+FqJxEH65yqQ/T16upqobveusCk2/XSeTW+/+GtWZA61suwb/Utyxfv8boYqczQ9QcmPTN9Jj7UtrbBWkQ+03TS3xEJXQh6WSTSrW5lB8srbSuH+GXyRtVITiRcMsSwEX647bT3sHjJPjsd1OzkTFRwEq6bXeUGE3Q3CUWwX//w42KxaNT2iJWoxBc10h+8aBGyJVH2alW8fvn9T0tGf8vyZsHwq1sJgQ1bwdfqL+WsLyLtkjXhAJZb+FxGSiRqaAMI15vESciLnI2zfR+fnOrsfXx6GWmQHJbsAEzYzKKknH7WaeBXtiTnGKNsfTa8rKQna4qjNdki3QyRdZLyLfsnuzSRNXq7VS6CegUzQgFPo3hhrCGXixl3Zi6PWkY/X6zmkn7OHiY7ktcnFEf+HOG/j1ZOd3rLjA2TI5U0h6I8xiZan4ML4a9OLlCrQZtBzcOBBFUS5KYpJlxWsAZhJa+lR0IAJiOdk4diVDvmGhQD4gNsrHmKP99UKSkFl77WmpcjFsLgTNa4ZI9cm34IYmwFnt+wVpk5BMsl4+hXz1ElKeXyxLvxx3P4T7jyGUSqrg8HMTpyJBI5JyDOAfyC+GboW42WpERsWUKi/naGqm4uIz99QUZsseLYc5MoMpTQlvNf+aEryvJpqlaq4KJR96q1PekTKO4lah74SI8pXEtqmXeC2lVGRjoHlqlnFektpQVwSq+VekDM/RxLaCFesbe6VW/i3RvnrCt4fKEBkAyiqdAjnMrB3YonsWQA1FHvrForG19EzUe7qtjQgnoIRTlZQtdrLtsdUB/2Hd+wb1ZI5zRhRBxDIg4RgdmB5/tXnKZC4kTfRvdD2wcKz934hDmhJg96NH5USIgj6YGmz8h6vXVKscsRkR3LweruGu0KMFHWVx/doEaUx5CsgG/AKgyUUXpy0MYBvqNvN5fenmOFx3Cljg369DCLIeT52uk+XNAIxKhVQlxUCKouNxuB6e2FNGgmabqC+61sPdo4T+VFJ49PT1NXjlMZk2gw1JnXBZ8hEb6d4RLWFXxv2wYR4zOQxvPj2Dq5micNWjJc6WY693ZwsSvPoEUU7Dsym4Jf0Q1/BmD061tpnmemm6+yAp9F0r4eczvis+tbRQkmJFJzHskAJb8XiDN5it4AHQehHmQd+OaiL4CXbEAf3av6rrdoBGNeo54El2cun5OUcXDpG0kazaPjScrZDLksmt/M6CgpQPpQILCmGVOCmOzUWCb0q/09zfdoQhHBw14wXuGYn0Q124rzE+tGEG7WkXcD3/39ZO5/WfshGUW7axqwaWWMFT8l7NJm2lmpeOfQRgXPsR2XUk/FPKL7ZnqcXc4gjvug/HT5lB2alBp0Hs/Ic2oTRUQBDNkczZptPGeLIIVtydaxUExZ+aC6WKyxRqPgmRi06UbXeZcmDJPWxydM46eJJjKDbxnX1biEjdorbJydP982ZkLXxLjJotMLFJxvmfHv0Sg0x52ilpmH1FfAcr7zFol3eerYaO5866PGcAmp1KcjUsQMXDdzSCKVz10lzV/YQM5+PCc+7S4xKSS2x/iKNySVInZMQjHFgqchK057WwpE7FCxlTX4+vBFosFnk8G0DKvXJYXwd8PnAKD1YsbyyPu99GRB7u5fsPWZPYyWk072BehOElDRF5kCTLC5gHWL3O8vJuSshZ500MX/5f20fhIRFir0HSHIPSHYCiYIgFMbIXgCd/wCdLBt/BqsfnG7ger/fbwp6DuYRi88XgnR2BrDdpqUJJcK52REXqMTUmOI4258SZ7SE7V3PiutZEElm0bIrLrg19ccXqXyjPs31T19HP45aGRgFmbwUHFnKfEfyUFa4nXeJemkulgo4+lsq4zv42oZX8jYcvEvUEsDBBQAAAAIADNiSV3wDsInRQIAACIGAAAhAAAAc2NyaXB0cy92ZXJpZnlfcHJvamVjdGlvbl9vbmx5LnB5jVTLbtswELz7KzankI1quEUPRQHl1mMfh7YXwzBYaWUzkUhiSTX233dJSrLiOK11ECByZzg7s5TunKUApu/cEZQH4xYN2Q7csbKVDda2HnSu6ZR/jCVxYxs/cmWjW9WhCUttl9TiWF3tsXrEestLixobBnqksK21f7DaBFHZ3gRfgN8rh6V4v/rwsYD4lvLTAvhhoIcS1jMiUQ31EhpLUIE2kHk2CaKbhMr4+Dil6Ul7ZB7jlsorInUUUwMsuRcRUUB+r1cbeAMtmrQqpZyYGM6Ntqxf7axRrRipC1idqljAtAH3vLNU5ijkSVBqTMXdX6rt8TORJXH77Q9Sq5zTZsct+aBMheD0Adl8RRib/a3rGs3yVi6SmwzQzXHryD5gFbQ1wpLeaRZWwLCI9T/MPU/jHJNb8ranCq8IYTw8xxAU7TBcAZtOzbjecCMM+2oN5jxNjQdeWKWvCEsDZgZhJ1ttNjCWxgwykfaJCdjFCAqz3NlUNfvskAWLNZMXGbspIgDJs7ryB/U4HwXCTmkTsyov0TKLlPB21DSfjQn5fCB4Z+i0TLOX/TubmsuT80V7H5UQel33qo0m4Y74TtRxVuZgVYVYUA7xrNORm+clrP9yUxks5blukSA35cwUTulqszPtJbcjZwS+8CDFTSCmjI0NOWdl6qtOTsj/nb2SV/n/PV5SsE+GGfbaRWWnBIY5rHXT8PZ5HjnyuxLeTcvjBUhTfnGMX2uGERs5/gMz880rw/SyiZ8GDy7dQ8BDIDX9g9Lv5i9QSwMEFAAAAAgAM2JJXQAAAAACAAAAAAAAABgAAABzcmMvZmlsYW1lbnQvX19pbml0X18ucHkDAFBLAwQUAAAACAAzYkldAAAAAAIAAAAAAAAAGwAAAHNyYy9maWxhbWVudC9pby9fX2luaXRfXy5weQMAUEsDBBQAAAAIADNiSV1dhJ7nEQgAAIkXAAAWAAAAc3JjL2ZpbGFtZW50L2lvL3JsZS5webVYbW/bOBL+nl/BywInaeuoSdotCqPul7YHFNiXw75+cA1BkSibF5nUknQSN5v/fs+Q1JvtONkDzggCieQMZ4bPPDPU6enpL1aLwrJ1bq5ffvjpw09nP3//6eWHX35nhZJW54U1EyZkyRuOf9IyVbG1KnmN+ZKnp6enJ5VWa5Zl1cZuNM8yJtaN0pblUiqbW6GkOTkJY4W5aR9XuVnV4qp9/Y9R0mtqcksTrZp/47WTl5t1s2W5YbIJi7eFKpRVqjatAHlCS2gio5eTk5OSV6xY8eKal5mueVyojbRmyozV7C92tbUcXppV3vAps5um5nMhLfltFwk7e89KhGh6wvCDw7/ntShzy1nJKQYl0xvJai6XdmXYFa+U5vBOlkIuGdblzCpmV5xJBOOmldIudKRRVCQcu90T9o8Zu2RKI3rb2G4bHsuECfirLFlDM5K9m7FzVrlHIb3ZibeOfjoXhjMYueGftFY6jn6giLhlbL0x1p1sDkl7q1ijjHBmQTtfcm3SKGnNEkZIY3NZtAGb+FANNvPjbBYeUu9cHOWmEKLX5Mzf14bwJ84lZYOCY258umt4YRFvqSRfN3YLmXWjuTEYI+i2RoR9cSzY44Y00OGKCidqAcka5s4XE3bu/9xaimaxyjUFdNeSAuuVLmOaBxrY67fdVHDtnI6koH9vXvVyh734LG8IP/u2QzuyDcAI5tPPGc/+mrG4YP9kry4S9u6dd6Vb4t7Yixn7bmiVH33PXp8/1x5nRI/koRXQ57a/HOsiGAm54bvrLt6M13VOnF3smx/QT2eVwN7LQ6JwjubnZ5eLoZyffMeedPFHvvSpd8RHDwzs5LQOt/Ez730Czc8X7NvweLF4amNQKeN3BeclsVO+RPKLNZeGOHG4OXmX5g1RbOz230FAgC9w2AG2O2MA11sI5jhq4qGjJxDWHFRGlipCP40gWFc1f9RizUH0kt1HRnzl0ZTVwthAXxMW+ezBcEhGLkeM8BDY2A87fo7p3xSUnsoy1zrfTti3gY1nPyLVJyyva3WbuZyf/SuvDXekDPbw3jnGn5GC3DgFTmNHPvQC1WLdkau3tuVV2gN8W3rmd7Jpz8Z+0L8fZVmKptY81NIDcQtkATOFqYQU1u+VpHAv7njQTQvppiZsjgO/WIQlT3G88fR+xZlX75y6EjLX2x0bBgF1i2jM+U1l5+g+n5xMy+ROyLAcFY9UgNCoUh4AS1eMWzy4wwLrosWQ/ZlNWElFb4bZDQrS2ySZt4Ba7NUWDyQ/6oH0eFmfxZfnr99OGP1PHL4OY6oHoQ8CmgUqbnutw2SMi2OxhR9x7z5ClcfQkyRsNhtS1/8h1CFgtFsI1k1oXbKBeDx4nrpOZ474TVxa09MCpdJRQSbKg+EMeOFteQYPcRt3Il1waHSwl8uuVuhIHKrosyOiQt1w7RmpqtCoTNlaGGTKcnZvgCNexq2ys92dkvn01eIB5DTi69GvijbyWqpb2WnbVXLWqvfaQtCpcWhdnQTOyyh01EkMDwn5uDbD3GrboiBM+R9lEYmNhl7uD335Mhp7ssK78H3+2NOD0mIJWqjBE6iE+ZojY2DdsCSBQdANNlr49o6CMZikbsl3fl2/5JwemxJ4eT9DA4qS0epv2IccnZ0ogNGvsEghk9AOSeq0qWY74oBVhq4ggu4itJQ14o6j+UcGK1hSA988HbcprVaYslt1xhZQ/9IthmPDGEz3oHMAqB/BCRC2fJy1Ll737XE9DMO8G+o0L9Fotka0aXurgZ7MbK4c4pWM6Y40GcLreSmKK8cfpIrR/a6pNwgbDJSi4sCFXeWW8hEeCFtvQSeoZ2hcvnKtzko0CW4nv4/pbi9PMcq+Yd558gAHQrc7500/mjbgOGnT9XUpdOxfzOxXTW0QvwPIMnXtXr0IgNtAkZO8FXaFMFWVuHNKU//MXrAotesmxJ1WObFUoeOKo9towiS/rYXksyhK6O5Y9cftYq+pCJib1L/EVbIzHSbUbTyPkFPIKGnhMhRHhi/pzV2EqYBEi3EedZlNVznPPEPWGePOCUwGucdxIwYropUYCM1bnWDui2QfuXs2V1GPzuxePETtFos+xqnmTZ3j8tafVgedGTWDdLZ73aBX2/WE9+jzqN2/SZwrqPc3h/1u6TJ5mOyZH+EkMuxw+d0bqAzfEVI/4I8dNbbMXPmHhnTF70oBzILBDigjeGcdvLNGAf9b6PUTCI/JhMxoyzY36L11PTqg8TpfLmukK9coWL4RyNo4+E8qpB7cn2FeVIKX0UOP/mMYbndN6WtJlPgTzCy/szGNpOVm3Zi4XeW/20g7u0xeRF/kuEtoFwWGcRHbI5ijnOK6KM3/3AjNu3j4xJweS/H+kAk3XRHDlXwvH7r9fXwM55LeRvXIZbMLlMvmxxOZXOwS+SNM+NkNDJNZVGFVilOpS6qMhrqUZyT109fs9nbgv63Vm/X4Dki+A23ktrdhRyN6SkG9FR7G5ix2a5i7z5AaaPNKW3pp4xdaDf9RKSx4RoHDJaOGmWv65OV1l13BG1i0W9/69ihAeRa2TLWBuI1hCxHVrh8dEoKdQ+SEy5JX+BzTf/PdXbjUfv5IGnyP7nXsGt3GyhVkb+54wUG+bS/x7pAeY/7/tWQ6dI7Trfd8wMWOCmqVlyb+24zieMARSjJKi3bdPPD8gtJiQPUUzn7NiPXd0keI/wDjPzyVST5/2ruA4ibcXm2x6kzY+XrVWzYoHc6uv1U9nmMZKXzKqkDBA+dP/gtQSwMEFAAAAAgAM2JJXQAAAAACAAAAAAAAACEAAABzcmMvZmlsYW1lbnQvdHJhaW5pbmcvX19pbml0X18ucHkDAFBLAwQUAAAACAAzYkldMWh6xAoGAADrDwAAIAAAAHNyYy9maWxhbWVudC90cmFpbmluZy9ydW50aW1lLnB5rVfdjxM3EH/PX+HeS3bvcqtcEH0AgigUJAQF1NKqEkUrZ3c2MfHaK9ubXLjyv3fGXm92QxBV1Tzcre3xfP7mwxcXF78JtZZwvdHWMdMqpvcKjN2IZsZK4KUUCizjqmScrXSrSihZZfRnUNcVcNcaYAUvNpBdXFxMRN1o41hVKCfjYsPtRopVXH6yWsVvbeOX1cUWXFw5UcMEhdSs0FJC4YRWlnWHb00JBsqfReEijXJw61BGJOl2aq74GkygarjbDEje4XIymZRQMbvhi/s/JkSQPpgw/BlAu1TUPOsI6Eqgygw6Jl8dHNgkTbMN3JZiDdYlaceSO12LIidb/Y0Z23HZQseedtiSHfn1u1nDDSiX1dtSmCQs7PK9aWHG4FZYl+utX4YrDsgYbg7IzV/fC7fJFa/B883oi12xaprdaZutwTWiTNIvmaub6QmHbG+Eg5zclpDaWdnWjU282ihbWQx0zm0hxPIFlxb3BEJBueUiRQnTv1THEOUYaCQvIOl5z1iwcjJ5chIZ8pUzXCjEYC4RA523EIe5KDt3IbBeATQoUJfArOMrCQ8RmxIcXkNYVnSTVUISFBXjUuo9c3vN/H7As4fn/+p88nW4pxtQyZRfTVPGLaJGlRKC6t7F5nBc0M9nR+aVTgJxRrornaSz7vD122ev8ud/sr+H6zdP054P3BbQOPaUmKATXr59bow2JB5PxuLQvRbYr62irPJkyfQnpd0GDDmIyz4C7JNesY2WpWV4yhqjP2HueS9maJtPI2T/bcs6ayzANpmn544cRrbgDpLxaY+45G6KGJ0+YAO8ztiUihNuhipBB7ThcU6nAS14Hj5mI9aD3xSxYxyUeavELZKTQzL6gzkx6zQ8q3UlW7s5UfkgQJb9TiUUwu4/Bvr3N5QbheTWsveoztO2RBOfY4gBq20yDF1fQaztrwTycEAJlecYTZfniQVZzZgFTLrSzlhBiiy90bVGAGglivSosagYbrI5exSvsEdLdu/efH4OT39QYejQ9E5I7XsH8WZ1i41khfDRVjixg9A7HKsxZMQtcs+mR4eSppnXD1PT/x8f+cDFo+TkXuxSeD4gvopyJp76CaK5AeMOvZsM1AH23k8DR3TVf8z7eqAjlfnIBTsfbgRPG7BgdrCcj73qL/bSyKkd4Tm/nol/51/v3JU/QgbUcsuHGDFWtcbn8l6bLesQTr6N8HgRuvQzutEX1JfquoZaY+fQSh6QD+yQhYHWYnvnhdF4ccWL7Uqj7VhYGizqRhdgLZmAtJZacl9Uz8AOyXeguCog3wK2gJrf5tqINSWKXS7Chm+hy/s3C3bJFpeXi/nXeLwhh40u08aClOo50M53UfpiOK30KI0jjdMRo75z9MK+QunYLmq8o40x8Ujv2XEvaH1i18AlYy5ro9uGyAejz2kSRI5+sRGu/66FtccT2Ilumuo2GuDbXp35EdeIsi6QosZWHWLosHqFL6k5qjJjl6eBHgfwxFk/LM/58LuB89MmG02bA86YglytA+YjE++xaGJwH3WNpLclHWoZqIX1eHtDiKeSFY3FuSNQjPUcsq5Rl9zpHFR5TkRP7uNytWQ3Y4tDvfG8PkSpH0+AFKI4uts5hALn2kZCQuXKoXeSNCualv7itCXWrW5xTGUVZkxFxoTY4eB6FCE+++rZ1shDtTVIpL9kCBcJNU5BOREMWUTRIzd6Lo9PQH4uuL/4yhNHkT6wKAjrEfiyFyaQEOpQ84bh3W9o1hvmBgUsGSTC1XltvjUZxB/adwygh4NQX8nBDp4M9lL2eHkm29N0bHpO/YGAUp7gstENTt51gsXahdn6DHKCVdf/NkaW9O7EZX6GHyGgsmdEHEvDCGVDVS04rA28lYNMmrG7L+kRtmhcBMdZFt9JlWEIlz6GJ9V3WK7Q34OYz04pjly7DOsVO3ZvCdyc9v+huoHgG7V23r328EkZYsDJfzk9G8nfcVQrtmhmfHLkOY2AeU5vSKvlDkPWvTvsh5uPw9fnnXX4CkEyyWmMQqclHS+EVv9oDVFtPE41tf5IlJm11Ktkepk1h2mafpn8A1BLAQIUAxQAAAAIADNiSV2pJPypfAYAAHYQAAAaAAAAAAAAAAAAAACAAQAAAABzY3JpcHRzL2Rpc2pvaW50X2V4cG9ydC5weVBLAQIUAxQAAAAIADNiSV3wDsInRQIAACIGAAAhAAAAAAAAAAAAAACAAbQGAABzY3JpcHRzL3ZlcmlmeV9wcm9qZWN0aW9uX29ubHkucHlQSwECFAMUAAAACAAzYkldAAAAAAIAAAAAAAAAGAAAAAAAAAAAAAAAgAE4CQAAc3JjL2ZpbGFtZW50L19faW5pdF9fLnB5UEsBAhQDFAAAAAgAM2JJXQAAAAACAAAAAAAAABsAAAAAAAAAAAAAAIABcAkAAHNyYy9maWxhbWVudC9pby9fX2luaXRfXy5weVBLAQIUAxQAAAAIADNiSV1dhJ7nEQgAAIkXAAAWAAAAAAAAAAAAAACAAasJAABzcmMvZmlsYW1lbnQvaW8vcmxlLnB5UEsBAhQDFAAAAAgAM2JJXQAAAAACAAAAAAAAACEAAAAAAAAAAAAAAIAB8BEAAHNyYy9maWxhbWVudC90cmFpbmluZy9fX2luaXRfXy5weVBLAQIUAxQAAAAIADNiSV0xaHrECgYAAOsPAAAgAAAAAAAAAAAAAACAATESAABzcmMvZmlsYW1lbnQvdHJhaW5pbmcvcnVudGltZS5weVBLBQYAAAAABwAHAAcCAAB5GAAAAAA=')
assert hashlib.sha256(raw).hexdigest() == '5f00a58a78794534e5e1fda27e553c4066cc9f58343a39aaa87919cff53ee577'
root = Path('/kaggle/working/disjoint_replay_source')
assert not root.exists(), 'Existing export source preserved'
root.mkdir()
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    for item in archive.infolist():
        path = root / item.filename
        assert not Path(item.filename).is_absolute() and '..' not in Path(item.filename).parts
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(archive.read(item))
sys.path.insert(0, str(root/'src'))
sys.path.insert(0, str(root/'scripts'))
from filament.io.rle import read_submission, write_submission
from filament.training.runtime import sha256
from disjoint_export import exclusive_masks
from verify_projection_only import verify_projection
if USE_CURRENT_RUN:
    run = Path('/kaggle/working/inference_run')
else:
    matches = [p.parent for p in Path('/kaggle/input').rglob('inference_run/report.json')
               if json.loads(p.read_text()).get('phase') == 'final']
    assert len(matches) == 1, 'Attach the completed public final-001 notebook output'
    run = matches[0]
source = json.loads((run/'report.json').read_text())
plan = json.loads((run/'plan.json').read_text())
assert source['status'] == 'completed' and source['fresh_session_replay_exact']
assert source['checkpoint_sha256'] == 'f99eb52031e3f8dc1011b64196b55b5887b9c85070e81695aee98553139fb6d4'
assert sha256(run/'submission.csv') == 'd18a13ccb8dc645f0a3f3899833df919669c91e9297b40079c8ce436f05843bf'
assert sha256(run/'plan.json') == source['plan_sha256']
names = sorted(plan['test_images']); assert len(names) == 180
original = read_submission(run/'submission.csv', [Path(n).stem for n in names])
predictions = {}
for name, masks in original.items():
    projected, _ = exclusive_masks(masks)
    verify_projection(masks, projected)
    predictions[name] = projected
    if len(predictions) % 20 == 0:
        print(json.dumps({'verified_images': len(predictions)}), flush=True)
output = Path('/kaggle/working/disjoint_output'); output.mkdir(exist_ok=False)
write_submission(output/'submission.csv', predictions, original)
report = {'status': 'completed', 'source_csv_sha256': sha256(run/'submission.csv'),
    'csv_sha256': sha256(output/'submission.csv'), 'original_images': len(predictions),
    'instances': sum(map(len,predictions.values())), 'no_overlapping_instance_pixels': True,
    'foreground_union_preserved': True, 'gpu_used': False, 'training_performed': False,
    'python': sys.version, 'numpy': np.__version__, 'pycocotools': importlib.metadata.version('pycocotools'),
    'seconds': time.monotonic()-started}
(output/'replay_report.json').write_text(json.dumps(report,indent=2)+'\n')
signal.setitimer(signal.ITIMER_REAL, 0)
print(json.dumps(report,indent=2))
